# Tuchanka design notebook

Design, preview, and export gcode for the Tuchanka (Prusa XL 5-tool) printer.

Run cells top to bottom with shift+enter. If you change a cell, re-run it and all cells below it.

In [ ]:
import fullcontrol as fc
from fullcontrol.devices.personal.tuchanka.setup import starting_steps, ending_steps, base_settings

In [ ]:
# printer / startup parameters

design_name = 'my_design'
initial_tool = 0
tools_used = [0]
first_layer_temps = [215.0] * 5   # per-tool, °C
idle_temps        = [70.0]  * 5   # per-tool standby, °C
bed_temp          = 60.0
print_area        = (20, 20, 180, 180)  # (x_min, y_min, x_max, y_max) mm

# set heat_soak=True or nozzle_clean=True to enable those startup stages
startup = starting_steps(
    initial_tool=initial_tool,
    tools_used=tools_used,
    first_layer_temps=first_layer_temps,
    idle_temps=idle_temps,
    bed_temp=bed_temp,
    print_area=print_area,
    heat_soak=False,
    nozzle_clean=False,
)

In [ ]:
# design parameters

EW = 0.4  # extrusion width (mm)
EH = 0.2  # extrusion height / layer height (mm)
initial_z = EH * 0.6  # slight squish for first layer adhesion
print_speed = 1000  # mm/min

In [ ]:
# generate the design — replace this with your actual design steps

layers = 5
design_steps = []
for layer in range(layers):
    design_steps.append(fc.Point(x=50, y=50, z=initial_z + layer * EH))
    design_steps.append(fc.Point(x=100, y=50))
    design_steps.append(fc.Point(x=100, y=100))
    design_steps.append(fc.Point(x=50, y=100))
    design_steps.append(fc.Point(x=50, y=50))

In [ ]:
# preview the design (startup gcode steps are excluded — they're not geometry)

fc.transform(design_steps, 'plot', fc.PlotControls(style='line', zoom=0.7))

# for a more realistic preview showing actual extrusion widths, uncomment:
# fc.transform(design_steps, 'plot', fc.PlotControls(style='tube', zoom=0.7, initialization_data={'extrusion_width': EW, 'extrusion_height': EH}))

In [ ]:
# generate and save gcode

shutdown = ending_steps(travel_speed=8000, present_print=True)
steps = startup + design_steps + shutdown

gcode_controls = fc.GcodeControls(
    printer_name='custom',
    save_as=design_name,
    initialization_data=base_settings(
        print_speed=print_speed,
        extrusion_width=EW,
        extrusion_height=EH,
        nozzle_temp=first_layer_temps[initial_tool],
        bed_temp=bed_temp,
    ),
)
gcode = fc.transform(steps, 'gcode', gcode_controls)

---
*Tuchanka notebook — Petricore Labs*